In [10]:
import numpy as np

In [11]:
X = np.array([1, 4, 2, 7, -3, 11])
y = np.array([1, -2.5, 4])
X_new = X.reshape(3, 2)
y_new = y.reshape(-1, 1)
X_new, y_new

(array([[ 1,  4],
        [ 2,  7],
        [-3, 11]]),
 array([[ 1. ],
        [-2.5],
        [ 4. ]]))

In [12]:
Tildae_X = np.hstack((X_new, y_new))
Tildae_X

np.linalg.det(Tildae_X)

np.float64(96.5)

In [16]:
import numpy as np
from numpy.linalg import inv, matrix_rank


def det_checker(X):
    m, d = X.shape
    if m == d:
        return "even"
    elif m > d:
        return "over"
    else:
        return "under"


def RC_checker(X, y):
    X_aug = np.append(X, y, axis=1)
    rankX = matrix_rank(X)
    rankX_ = matrix_rank(X_aug)
    d = X.shape[1]
    if rankX == rankX_:
        RC = 1 if rankX == d else 3
    else:
        RC = 2
    return RC, rankX, rankX_


def evenSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X) @ y, "Unique solution."
    elif RC == 2:
        return None, "No solution."
    else:
        return None, "Infinitely many solutions."


def overSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X.T @ X) @ X.T @ y, "Unique solution."
    elif RC == 3:
        return None, "Infinitely many solutions."
    else:
        return inv(X.T @ X) @ X.T @ y, "No exact sol, least square approx."


def underSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 2:
        return None, "No solution."
    else:
        return X.T @ inv(X @ X.T) @ y, "No exact sol, least norm approx."


def solveLE(X, y):
    det = det_checker(X)
    if det == "even":
        w, ans = evenSolver(X, y)
    elif det == "over":
        w, ans = overSolver(X, y)
    else:
        w, ans = underSolver(X, y)
    print(ans, "\nw =", w)
    return w


# --- Quick diagnostic ---
X = np.array([[1, 3], [1, 4], [1, 5], [1, 6], [1, 7]])
y = np.array([[5], [4], [3], [2], [1]])

print("System type:", det_checker(X))
RC, rX, rX_ = RC_checker(X, y)
print(f"rank(X)={rX}, rank(X~)={rX_}, case={RC}")
w = solveLE(X, y)

System type: over
rank(X)=2, rank(X~)=2, case=1
Unique solution. 
w = [[ 8.]
 [-1.]]


In [18]:
X = np.array([1, 3, -2, -4, 0, -1, 3, 1, 8, 2, 1, 6, 8, 4, 6])
X_proper = X.reshape(5, 3)
y = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1]])
bias = np.ones((X_proper.shape[0], 1))
X_b = np.hstack((bias, X_proper))  # [1, x]
W = solveLE(X_b, y)

No exact sol, least square approx. 
w = [[-0.01005025 -6.03015075  7.04020101]
 [-0.24874372 -1.74623116  1.99497487]
 [ 0.44723618  3.34170854 -3.78894472]
 [ 0.03015075  1.09045226 -1.12060302]]


In [19]:
x_new = np.array([[1, -2, 4]])
x_new_b = np.hstack((np.ones((1, 1)), x_new))
y_new = x_new_b @ W  # two predictions
print("y_new =", y_new)

y_new = [[ -1.03266332 -10.09798995  12.13065327]]


In [24]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import inv
import numpy as np


def polyTx(X, order):
    """Polynomial feature matrix with bias column. Shape: (N, order+1)."""
    return PolynomialFeatures(order).fit_transform(X)


def solvePR(P, y, ridge=False, lamb=0.01):
    """Solve polynomial regression. Auto primal (N>M) or dual (N<M)."""
    if ridge:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P + lamb * np.eye(P.shape[1])) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T + lamb * np.eye(P.shape[0])) @ y
    else:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T) @ y
    return w


def solveLE_Ridge(X, y, lamb=0.01):
    """Linear regression with Ridge. X must already include bias column."""
    return solvePR(X, y, ridge=True, lamb=lamb)


# Quick test
X_test_poly = np.array([[-10], [-8], [-3], [-1], [2], [8]])
y_test_poly = np.array([[5], [5], [4], [3], [2], [2]])
P = polyTx(X_test_poly, 3)
print("P shape:", P.shape)  # (6, 4): [1, x, x^2, x^3]
w = solvePR(P, y_test_poly)
print("w:", w.ravel())

P shape: (6, 4)
w: [ 2.68935636 -0.37722517  0.01343815  0.00285772]


In [25]:
X = np.array([1, 3, -2, -4, 0, -1, 3, 1, 8, 2, 1, 6, 8, 4, 6])
X_proper = X.reshape(5, 3)
y = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1]])

In [29]:
P = polyTx(X_proper, 2)
P, P.shape

(array([[ 1.,  1.,  3., -2.,  1.,  3., -2.,  9., -6.,  4.],
        [ 1., -4.,  0., -1., 16., -0.,  4.,  0., -0.,  1.],
        [ 1.,  3.,  1.,  8.,  9.,  3., 24.,  1.,  8., 64.],
        [ 1.,  2.,  1.,  6.,  4.,  2., 12.,  1.,  6., 36.],
        [ 1.,  8.,  4.,  6., 64., 32., 48., 16., 24., 36.]]),
 (5, 10))

## FITB

In [ ]:
X = np.array([4, 7, 10, 2, 3, 9]).reshape(2, -1)
y = np.array([-1, 1]).reshape(2, -1)
X, y

(array([[ 4,  7, 10],
        [ 2,  3,  9]]),
 array([[-1],
        [ 1]]))

In [35]:
import numpy as np
from numpy.linalg import inv, matrix_rank


def det_checker(X):
    m, d = X.shape
    if m == d:
        return "even"
    elif m > d:
        return "over"
    else:
        return "under"


def RC_checker(X, y):
    X_aug = np.append(X, y, axis=1)
    rankX = matrix_rank(X)
    rankX_ = matrix_rank(X_aug)
    d = X.shape[1]
    if rankX == rankX_:
        RC = 1 if rankX == d else 3
    else:
        RC = 2
    return RC, rankX, rankX_


def evenSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X) @ y, "Unique solution."
    elif RC == 2:
        return None, "No solution."
    else:
        return None, "Infinitely many solutions."


def overSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X.T @ X) @ X.T @ y, "Unique solution."
    elif RC == 3:
        return None, "Infinitely many solutions."
    else:
        return inv(X.T @ X) @ X.T @ y, "No exact sol, least square approx."


def underSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 2:
        return None, "No solution."
    else:
        return X.T @ inv(X @ X.T) @ y, "No exact sol, least norm approx."


def solveLE(X, y):
    det = det_checker(X)
    if det == "even":
        w, ans = evenSolver(X, y)
    elif det == "over":
        w, ans = overSolver(X, y)
    else:
        w, ans = underSolver(X, y)
    print(ans, "\nw =", w)
    return w

print("System type:", det_checker(X))
RC, rX, rX_ = RC_checker(X, y)
print(f"rank(X)={rX}, rank(X~)={rX_}, case={RC}")
w = solveLE(X, y)

System type: under
rank(X)=2, rank(X~)=2, case=3
No exact sol, least norm approx. 
w = [[-0.21052632]
 [-0.47368421]
 [ 0.31578947]]


In [ ]:
X = np.array([4, 7, 10, 2, 3, 9]).reshape(-1, 1)
y = np.array([-1, -1, -1, 1, 1, 1]).reshape(-1, 1)
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))
W = solveLE(X_b, y)  # W shape (M+1, 2)
x_new = X_b
y_new = x_new @ W  # two predictions
print("y_new =", y_new)
# ans: 2

No exact sol, least square approx. 
w = [[ 0.74468085]
 [-0.12765957]]
y_new = [[ 0.23404255]
 [-0.14893617]
 [-0.53191489]
 [ 0.4893617 ]
 [ 0.36170213]
 [-0.40425532]]


### Original input features

In [41]:
x_new = np.array([1, 6])
y_new = x_new @ W
y_new

array([-0.0212766])

### 4th order

In [42]:
X = np.array([4, 7, 10, 2, 3, 9]).reshape(-1, 1)
y = np.array([-1, -1, -1, 1, 1, 1]).reshape(-1, 1)

In [45]:
order = 4
P = polyTx(X, order)  # shape (N, 4): [1, x, x^2, x^3]
print("NOTICE: PolynomialFeatures column order may differ from handwritten version!")
print("P:\n", P)

# Fit
w_poly = solvePR(P, y)
print("Coefficients:", w_poly.ravel())

# Predict on new point
x_test = X
# bias = np.ones((x_test.shape[0], 1))
# x_test_b = np.hstack((bias, x_test))
P_test = polyTx(x_test, order)  # same order
y_pred_poly = P_test @ w_poly
print("Prediction:", y_pred_poly)

NOTICE: PolynomialFeatures column order may differ from handwritten version!
P:
 [[1.000e+00 4.000e+00 1.600e+01 6.400e+01 2.560e+02]
 [1.000e+00 7.000e+00 4.900e+01 3.430e+02 2.401e+03]
 [1.000e+00 1.000e+01 1.000e+02 1.000e+03 1.000e+04]
 [1.000e+00 2.000e+00 4.000e+00 8.000e+00 1.600e+01]
 [1.000e+00 3.000e+00 9.000e+00 2.700e+01 8.100e+01]
 [1.000e+00 9.000e+00 8.100e+01 7.290e+02 6.561e+03]]
Coefficients: [-11.21971831  13.52136821  -4.88054326   0.65537223  -0.02923541]
Prediction: [[-0.76338028]
 [-1.11830986]
 [-1.04225352]
 [ 1.07605634]
 [ 0.74647887]
 [ 1.10140845]]


In [47]:
x_test = np.array([6]).reshape(-1, 1)
# bias = np.ones((x_test.shape[0], 1))
# x_test_b = np.hstack((bias, x_test))
P_test = polyTx(x_test, order)  # same order
y_pred_poly = P_test @ w_poly
print("Prediction:", y_pred_poly)

Prediction: [[-2.11975855]]


### Question 2

In [52]:
import numpy as np

# =============================================================================
# SHAPE CONVENTION — applies to every feature-selection helper below
# =============================================================================
#   X : np.ndarray of shape (N, M)   -> N datapoints (rows), M features (cols)
#   y : np.ndarray of shape (N, 1)   -> target, one value per datapoint
#                                       (a 1-D (N,) vector is also accepted)
#
# WATCH OUT: the Tut 7 PDF tables list features as ROWS (3 features × 5 points).
# If you build a numpy array straight from the PDF, TRANSPOSE IT first:
#     X = data_from_pdf[:3].T   # (3, 5) -> (5, 3)
# Otherwise you'll compute "correlations across datapoints" instead of
# "correlation of each feature with y".
# =============================================================================


def _check_X_y(X, y, *, name):
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    if X.ndim != 2:
        raise ValueError(
            f"{name}: X must be 2-D with shape (N, M); got shape {X.shape}"
        )
    if X.shape[0] != y.size:
        raise ValueError(
            f"{name}: len(y)={y.size} must equal X.shape[0]={X.shape[0]}. "
            f"Did you forget to transpose? If the PDF listed features as rows, "
            f"use X = data[:M].T so rows=datapoints, cols=features."
        )
    return X, y


def pearson_corr_manual(a, b):
    """
    Pearson's r between two 1-D vectors, using the biased (1/N) estimator
    per the EE2211 Tut 7 definition.
    """
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    if a.size != b.size:
        raise ValueError(f"pearson_corr_manual: len(a)={a.size} != len(b)={b.size}")
    a_bar, b_bar = a.mean(), b.mean()
    cov = np.mean((a - a_bar) * (b - b_bar))
    sa = np.sqrt(np.mean((a - a_bar) ** 2))
    sb = np.sqrt(np.mean((b - b_bar) ** 2))
    return cov / (sa * sb)


def feature_pearson_scores(X, y):
    """|Pearson r| between each FEATURE COLUMN of X and target y. Shape (M,)."""
    X, y = _check_X_y(X, y, name="feature_pearson_scores")
    return np.array([abs(pearson_corr_manual(X[:, j], y)) for j in range(X.shape[1])])


def select_top_k_features(X, y, k):
    """
    Top-k feature selection by |Pearson r| with y.

    Returns
    -------
    X_selected : (N, k) array   columns of X indexed by top_idx
    top_idx    : (k,) int array column indices of X, sorted by |r| descending
    scores     : (M,) array     |r| for every column of X (unsorted)
    """
    X, y = _check_X_y(X, y, name="select_top_k_features")
    if not (1 <= k <= X.shape[1]):
        raise ValueError(f"select_top_k_features: k={k} out of range 1..{X.shape[1]}")
    scores = feature_pearson_scores(X, y)
    top_idx = np.argsort(-scores)[:k]
    return X[:, top_idx], top_idx, scores


# Quick sanity check
a = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
b = np.array([2.0, 4.0, 6.0, 8.0, 9.0])
print("pearson_corr_manual:", pearson_corr_manual(a, b))
print("np.corrcoef (same) :", np.corrcoef(a, b)[0, 1])

pearson_corr_manual: 0.9938837346736189
np.corrcoef (same) : 0.9938837346736188


In [53]:
data = np.array([3.3459, 1.0893, 3.2103, 1.744, 1.6762, 
2.7435, 2.9113, 1.4706, 1.2895, 2.1366, -1.7253, -0.7804, -0.9944, 0.5307, -1.0502, 2.9972, 1.1399, 2.228, 0.3387, 2.5042])

data = data.reshape(4, -1)
X = data[:3].T  # (5, 3) — rows = datapoints, cols = features
y = data[3].reshape(-1, 1)  # (5, 1)

# |r| scores per feature
scores = feature_pearson_scores(X, y)
for i, s in enumerate(scores, start=1):
    r_signed = pearson_corr_manual(X[:, i - 1], y)
    print(f"Feature {i}: r = {r_signed:+.4f}   |r| = {s:.4f}")

# Top-2 selection
X_sel, top_idx, _ = select_top_k_features(X, y, k=2)
print("\nTop-2 feature indices (0-based):", top_idx)
print("=> Features", (top_idx + 1).tolist(), "are the answer to Q1.")
print("Selected X shape:", X_sel.shape)

Feature 1: r = +0.6510   |r| = 0.6510
Feature 2: r = +0.3722   |r| = 0.3722
Feature 3: r = -0.9308   |r| = 0.9308

Top-2 feature indices (0-based): [2 0]
=> Features [3, 1] are the answer to Q1.
Selected X shape: (5, 2)


## Q3

In [55]:
np.sin(6)

3 - np.sin(6) * 0.1

np.float64(3.0279415498198925)

### Q4

In [58]:
def gd_1d(grad_fn, x0, lr, num_iters):
    """1D GD: returns trajectory array of length num_iters."""
    x = float(x0)
    traj = np.empty(num_iters)
    for i in range(num_iters):
        x = x - lr * grad_fn(x)
        traj[i] = x
    return traj


# --- 2D scalar gradient descent (from A3 Task C) ---
def gd_2d(grad_fn, u0, v0, lr, num_iters):
    """2D GD: grad_fn(u,v) -> (du, dv). Returns (u_traj, v_traj)."""
    u, v = float(u0), float(v0)
    ut, vt = np.empty(num_iters), np.empty(num_iters)
    for i in range(num_iters):
        gu, gv = grad_fn(u, v)
        u, v = u - lr * gu, v - lr * gv
        ut[i], vt[i] = u, v
    return ut, vt


# ── 2D basin counting (A3 Task C pattern) ──
# f(u,v) = sin(u)cos(v) + 0.1(u^2+v^2)
f_C = lambda u, v: u**2 + u * v**2
grad_C = lambda u, v: (
    2 * u + v**2,
    2 * u * v,
)
pairs = [[3, 2]]

final_f_C = {}
for p in pairs:
    ut, vt = gd_2d(grad_C, p[0], p[1], lr=0.2, num_iters=1)
    print(f"ut:{ut}, vt:{vt}")
    final_f_C[str(p)] = float(f_C(ut[-1], vt[-1]))

rounded = {k: round(v, 1) for k, v in final_f_C.items()}
num_basins = len(set(rounded.values()))
print(f"Distinct basins found: {num_basins}  (rounded f values: {rounded})")

ut:[1.], vt:[-0.4]
Distinct basins found: 1  (rounded f values: {'[3, 2]': 1.2})


### Question 6

In [60]:
def mse(y):
    y = np.asarray(y, dtype=float)
    return 0.0 if len(y) == 0 else float(np.mean((y - y.mean()) ** 2))

In [62]:
def regression_split(x, y, threshold):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    L, R = y[x <= threshold], y[x > threshold]
    return dict(
        threshold=float(threshold),
        root_mse=mse(y),
        left=dict(n=len(L), mean=float(L.mean()) if len(L) else np.nan, mse=mse(L)),
        right=dict(n=len(R), mean=float(R.mean()) if len(R) else np.nan, mse=mse(R)),
        weighted_mse=(len(L) * mse(L) + len(R) * mse(R)) / len(y),
    )


data = np.array(
    [
        [0.2, 2.1],
        [0.7, 1.5],
        [1.8, 5.8],
        [2.2, 6.1],
        [3.7, 9.1],
        [4.1, 9.5],
        [4.5, 9.8],
        [5.1, 12.7],
        [6.3, 13.8],
        [7.4, 15.9],
    ]
).T
x = data[0]
y = data[1]
regression_split(x, y, threshold=3.0)

{'threshold': 3.0,
 'root_mse': 20.6381,
 'left': {'n': 4, 'mean': 3.875, 'mse': 4.3618749999999995},
 'right': {'n': 6, 'mean': 11.800000000000002, 'mse': 6.366666666666667},
 'weighted_mse': 5.56475}

### Question 7

In [ ]:
# ans: 200
# classifier 

In [69]:
books = np.array([50, 60, 66, 68, 71, 72, 75, 82,90,99])
A = books[:4]
B = books[4:]
np.mean(A), np.mean(B)

(np.float64(61.0), np.float64(81.5))